# R Master v0 · 手机版一键云跑

这个 Colab 用云端 Blender 4.4.3 跑 **Mona → Lapine 第一轮比例预览**。

你在手机上只需要：

1. 点 **运行全部 / Run all**。
2. 出现上传框时，选择 **`Mona.blend`**。
3. 等它自动完成。最后会显示正面 / 侧面 / 3/4 三张预览，并下载结果 ZIP。

### 这一步不会做什么

- 不修改原始 `Mona.blend`。
- 不改 RED 正式网页。
- 不烘焙最终 Rest Pose。
- 不做髋宽 -34% 的高风险修改。
- 不导出最终 VRM。

> 首次运行会下载约 350 MB 的官方 Blender Linux 包，属于正常现象。Mona 上传约 130 MB。请保持页面打开，最好在 Wi‑Fi 下运行。


In [ ]:
# R Master v0 · Mobile One-Click Cloud Runner
# Pinned engineering script commit: a1479f91b0a16de1dfe48712e4c8648a301ea636

from google.colab import files
from IPython.display import display, Image, Markdown
from pathlib import Path
import os, shutil, subprocess, urllib.request, zipfile, json, time

WORK = Path('/content/r_master_v0')
OUT = WORK / 'output'
BLENDER_ARCHIVE = WORK / 'blender-4.4.3-linux-x64.tar.xz'
BLENDER_DIR = WORK / 'blender-4.4.3-linux-x64'
SCRIPT = WORK / 'R_Master_BuildPreview_v0.py'
MONA = WORK / 'Mona.blend'
RESULT_ZIP = WORK / 'R_Master_v0_Result.zip'

BLENDER_URL = 'https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz'
SCRIPT_URL = 'https://raw.githubusercontent.com/hexiangyu481-commits/-erdan-lab-mobile/a1479f91b0a16de1dfe48712e4c8648a301ea636/red-r-master/blender/R_Master_BuildPreview_v0.py'

WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print('R Master v0 · 手机云跑')
print('① 请在下面弹出的选择器里选 Mona.blend')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('没有收到文件。请重新运行这一格并选择 Mona.blend。')

# Take the uploaded .blend even if iOS changed the visible upload order.
blend_names = [name for name in uploaded.keys() if name.lower().endswith('.blend')]
if len(blend_names) != 1:
    raise RuntimeError(f'需要且只需要 1 个 .blend 文件，当前收到：{list(uploaded.keys())}')
source_name = blend_names[0]
MONA.write_bytes(uploaded[source_name])
print(f'✓ Mona 已收到：{source_name} · {MONA.stat().st_size/1024/1024:.1f} MiB')

print('\n② 准备 Blender 运行环境…')
subprocess.run(
    ['apt-get','update','-qq'],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
subprocess.run(
    ['apt-get','install','-y','-qq','xvfb','libgl1','libx11-6','libxi6','libxrender1','libxfixes3','libxkbcommon0','libsm6'],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

if not BLENDER_DIR.exists():
    if not BLENDER_ARCHIVE.exists():
        print('下载官方 Blender 4.4.3（约 350 MB）…')
        subprocess.run(['wget','-q','--show-progress','-O',str(BLENDER_ARCHIVE),BLENDER_URL], check=True)
    print('解压 Blender…')
    subprocess.run(['tar','-xf',str(BLENDER_ARCHIVE),'-C',str(WORK)], check=True)

BLENDER = BLENDER_DIR / 'blender'
if not BLENDER.exists():
    raise RuntimeError('Blender 解压后没有找到可执行文件。')
print('✓ Blender 就绪')

print('\n③ 获取已锁定版本的 R Master v0 构建脚本…')
urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
print('✓ 构建脚本就绪')

# Clean only previous generated output, never the uploaded Mona source.
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

print('\n④ 云端 Blender 正在构建比例预览…')
cmd = [
    'xvfb-run','-a',str(BLENDER),
    '--background',str(MONA),
    '--python',str(SCRIPT),
    '--','--out',str(OUT)
]
log_path = OUT / 'R_Master_v0_blender.log'
with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        if ('R Master v0' in line) or ('Fra:' in line) or ('Saved:' in line) or ('Error' in line):
            print(line.rstrip())
    code = proc.wait()

if code != 0:
    tail = log_path.read_text(encoding='utf-8', errors='replace')[-6000:]
    print('\n--- Blender 日志末尾 ---\n' + tail)
    raise RuntimeError(f'Blender 运行失败，退出码 {code}。把这一屏截图发给二蛋即可。')

report_path = OUT / 'R_Master_v0_report.json'
preview_blend = OUT / 'R_Master_Align_v0_PREVIEW.blend'
front = OUT / 'R_Master_v0_front.png'
side = OUT / 'R_Master_v0_side.png'
threeq = OUT / 'R_Master_v0_three_quarter.png'
required = [report_path, preview_blend, front, side, threeq]
missing = [p.name for p in required if not p.exists()]
if missing:
    tail = log_path.read_text(encoding='utf-8', errors='replace')[-6000:]
    print('\n--- Blender 日志末尾 ---\n' + tail)
    raise RuntimeError('Blender 已退出，但缺少输出：' + ', '.join(missing))

report = json.loads(report_path.read_text(encoding='utf-8'))
print('\n✓ R Master v0 BUILD_OK')
print('  骨骼数：', report.get('bone_count'))
print('  复制 Mesh：', report.get('duplicated_mesh_count'))
print('  髋宽修改：', report.get('hip_width_change_applied'))
print('  Rest Pose 烘焙：', report.get('rest_pose_baked'))

print('\n⑤ 预览：')
for title, path in [('正面', front), ('侧面', side), ('3/4', threeq)]:
    display(Markdown(f'### {title}'))
    display(Image(filename=str(path), width=480))

print('\n⑥ 打包结果…')
if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in OUT.rglob('*'):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT))

print(f'✓ 结果包：{RESULT_ZIP.name} · {RESULT_ZIP.stat().st_size/1024/1024:.1f} MiB')
print('正在发送到手机下载…如果浏览器弹出下载确认，点允许即可。')
files.download(str(RESULT_ZIP))
